# Question 1
* Assume that `required_policemen` is a 6-dimensional input vector representing the minimum number of policemen required in shifts `1, 2, ..., 6`
* The function `solve_scheduling_problem` must return `optimal_cost, optimal_solution` where
    * `optimal_cost` is the optimal total number of policemen required
    * `optimal_solution` is a 6-dimensional vector such that `optimal_solution[i]` = number of policemen who come on duty at the beginning of shift `i`

In [8]:
using JuMP, GLPK

function solve_scheduling_problem(required_policemen::Vector)
    
    # create an optimization model using the GLPK solver
    lpmodel = Model(GLPK.Optimizer)
    
    # declare variables
    @variable(lpmodel, x[1:6] >=0)
    c = [1, 1, 1, 1, 1, 1]
    
    # declare objective function
    @objective(lpmodel, Min, c'*x)
    
    # specify constraints
    @constraint(lpmodel, RP1, x[1] + x[6] >= required_policemen[1])
    @constraint(lpmodel, RP[i=2:6], x[i-1] + x[i] >= required_policemen[i])
    
    # solve model
    optimize!(lpmodel)
    
    # if status is not `OPTIMAL` then something went wrong!
    if termination_status(lpmodel) != OPTIMAL
        optimal_cost = +Inf
        optimal_solution = nothing
        @warn("The scheduling model was not solved correctly!")
    else
        optimal_cost = objective_value(lpmodel)
        optimal_solution = value.(x)
    end
    
    return optimal_cost, optimal_solution
end

solve_scheduling_problem (generic function with 1 method)

### If you implemented your function correctly, then the following command should optimally solve HW1, question 2

In [9]:
optimal_cost, optimal_solution = solve_scheduling_problem([15, 35, 65, 80, 40, 25])

(140.0, [15.0, 20.0, 45.0, 35.0, 25.0, 0.0])

# Question 2
* Assume that the inputs to the function `solve_investment_problem` are as follows:
    * `budget` is a floating point number representing the investment budget
    * 'return_levels' and 'risk_levels' are 6-dimensional input vectors representing the returns (expressed as fractions) and risks of the following investments in order:
    `first mortgages, second mortgages, personal loans, commercial loans, government securities, savings`.
* The function must return `optimal_solution` a 6-dimensional vector such that `optimal_solution[i]` = optimal amount to be invested in investment `i`

In [10]:
using JuMP, GLPK

function solve_investment_problem(budget, return_levels, risk_levels)
    
    # create an optimization model using the GLPK solver
    lpmodel = Model(GLPK.Optimizer)
    
    # declare variables
    @variable(lpmodel, x[1:6] >=0)
    b = budget
    c = return_levels
    r = risk_levels
    
    # declare objective function
    @objective(lpmodel, Max, (c'*x))
    
    # specify constraints
    @constraint(lpmodel, TotalBudget, sum(x) == b)
    @constraint(lpmodel, AvgRisk, sum((r[i]-5)*x[i] for i in 1:5) <= 0)
    @constraint(lpmodel, CommLoans, x[4] - 0.2*sum(x[i] for i in 1:5) >= 0)
    @constraint(lpmodel, CombInv, x[1] - (x[2]+x[3]) >= 0)
    
    # solve model
    optimize!(lpmodel)
    #
    # if status is not `OPTIMAL` then something went wrong!
    #
    if termination_status(lpmodel) != OPTIMAL
        optimal_solution = nothing
        @warn("The investment model was not solved correctly!")
    else  
        optimal_solution = value.(x)
        optimal_solution = round.(optimal_solution ; digits=2)
    
    end
    
    return optimal_solution
end

solve_investment_problem (generic function with 1 method)

### If you implemented your function correctly, then the following command should optimally solve HW2, question 1

In [11]:
budget = 1e6
return_levels = [0.09, 0.12, 0.15, 0.08, 0.06, 0.03]
risk_levels = [3, 6, 8, 2, 1, 0]
optimal_solution = solve_investment_problem(budget, return_levels, risk_levels)

6-element Vector{Float64}:
 400000.0
      0.0
 400000.0
 200000.0
      0.0
      0.0

# Question 3
* Assume that the inputs `A` and `b` are the constraint matrices and right-hand side coefficients of an inequality form polyhedron: $P = \{x \in R^n: Ax >= b\}$
* Assume that `u `is an arbitrary n-dimensional vector
* The function must return `true` if `u` is a basic solution and `false` otherwise

# Hints
* Loop over all rows of `A`: if the [dot product](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.dothttps://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.dot) of `A[i,:]` and `u` is [approximately equal](https://docs.julialang.org/en/v1/base/math/#Base.isapproxhttps://docs.julialang.org/en/v1/base/math/#Base.isapprox) to `b[i]`, then the ith constraint is active at `u`
* Collect the indices of all such active constraints in an array `active_set`
    * For example, if constraints 2 and 5 are active at `u`, then `active_set = [2, 5]`
* Collect the rows of all active constraints in a new matrix `M = A[active_set, :]` and check if `rank(M)` is equal to `n` using the [rank function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.rank)

In [12]:
using LinearAlgebra

function is_basic_solution(A, b, u)
    m = size(A, 1)
    n = size(A, 2)
    
    # check if inputs are consistent
    if length(b) != m || length(u) != n
        @warn("Input data is inconsistent!")
        return false
    end
    
    # checking active constraints
    active_set = []
    for i in 1:m
        if dot(A[i,:] , u) ≈ b[i]
            [i]
            push!(active_set , i)
        end
    end
    
    # check for linear independent constraints
    M = A[active_set,:]
    rank(M) == n
        
end

is_basic_solution (generic function with 1 method)

### Test your function!

In [13]:
#
# This is the A and b for Example 1 from Lecture 7
#
A = [ 1.0 -1.0;
     -1.0 -1.0;
      1.0  0.0;
      0.0  1.0]
b = [-1.0, -2.0, 0.0, 0.0]

#
# feel free to add more checks!
#
if !is_basic_solution(A, b, [-1.0, 0.0])
    @warn("the function is not implemented correctly!")
end
if is_basic_solution(A, b, [-1.0, -1.0])
    @warn("the function is not implemented correctly!")
end
if !is_basic_solution(A, b, [0.0, 1.0])
    @warn("the function is not implemented correctly!")
end
if is_basic_solution(A, b, [1.0, 0.0])
    @warn("the function is not implemented correctly!")
end


In [14]:
# checking if function returns as "true" a basic solution [-1.0, 0.0]
is_basic_solution(A, b, [-1.0, 0.0])

true

In [15]:
# checking if function returns as "false" a not basic solution [-1.0, -1.0]
is_basic_solution(A, b, [-1.0, -1.0])

false